In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import spearmanr

# ── Local Paths ───────────────────────────────────────────────────────────────
PROJECT_ROOT = '/Users/pravintakpire/datascience/KAGGLE_COMPETITION/Predict_Customer_Churn'
CSV_DIR      = PROJECT_ROOT          # all existing CSVs are here
SUB_DIR      = PROJECT_ROOT          # new blend CSVs saved here too

print(f'CSV dir    : {CSV_DIR}')
print(f'Output dir : {SUB_DIR}')
print(f'pandas     : {pd.__version__}')
print(f'numpy      : {np.__version__}')

CSV dir    : /Users/pravintakpire/datascience/KAGGLE_COMPETITION/Predict_Customer_Churn
Output dir : /Users/pravintakpire/datascience/KAGGLE_COMPETITION/Predict_Customer_Churn
pandas     : 2.2.3
numpy      : 1.26.4


In [2]:
# ── Load Submissions ──────────────────────────────────────────────────────────
SUBMISSIONS = [
    {
        'file':         'submission_xgb_scaledpos_blend.csv',
        'label':        'XGB-scaledpos',
        'public_score': 0.91445,
    },
    {
        'file':         'submission_lgbm_tuned_multiseed_fe.csv',
        'label':        'LGB-tuned-FE',
        'public_score': 0.91419,
    },
    {
        'file':         'submission_xgboost_tuned_multiseed_fe.csv',
        'label':        'XGB-50t-FE',
        'public_score': 0.91417,
    },
]

dfs = {}
for s in SUBMISSIONS:
    path = os.path.join(CSV_DIR, s['file'])
    df   = pd.read_csv(path)
    dfs[s['label']] = df
    print(f"  {s['label']:20s} | score={s['public_score']} | shape={df.shape} | "
          f"pred range=[{df['Churn'].min():.4f}, {df['Churn'].max():.4f}]")

# Use first df as template for customer IDs
base = dfs[SUBMISSIONS[0]['label']][['customerID']].copy()
print(f'\nRows: {len(base):,}')

  XGB-scaledpos        | score=0.91445 | shape=(254655, 2) | pred range=[0.0001, 0.9926]
  LGB-tuned-FE         | score=0.91419 | shape=(254655, 2) | pred range=[0.0001, 0.9873]
  XGB-50t-FE           | score=0.91417 | shape=(254655, 2) | pred range=[0.0001, 0.9890]


KeyError: "None of [Index(['customerID'], dtype='object')] are in the [columns]"

In [ ]:
# ── Correlation Analysis ──────────────────────────────────────────────────────
labels = [s['label'] for s in SUBMISSIONS]
preds  = np.column_stack([dfs[l]['Churn'].values for l in labels])

# Pearson correlation
corr_df = pd.DataFrame(np.corrcoef(preds.T), index=labels, columns=labels)

# Spearman correlation
spearman_matrix = np.zeros((len(labels), len(labels)))
for i in range(len(labels)):
    for j in range(len(labels)):
        spearman_matrix[i, j] = spearmanr(preds[:, i], preds[:, j])[0]
spearman_df = pd.DataFrame(spearman_matrix, index=labels, columns=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr_df, annot=True, fmt='.5f', cmap='coolwarm',
            vmin=0.95, vmax=1.0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Pearson Correlation', fontsize=13, fontweight='bold')

sns.heatmap(spearman_df, annot=True, fmt='.5f', cmap='coolwarm',
            vmin=0.95, vmax=1.0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Spearman Correlation', fontsize=13, fontweight='bold')

plt.suptitle('Prediction Correlations — Lower = More Diversity', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nPearson Correlation:')
display(corr_df.round(5))
print('\nSpearman Correlation:')
display(spearman_df.round(5))

In [ ]:
# ── Prediction Distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(labels), figsize=(15, 4), sharey=False)

colors = ['#2196F3', '#4CAF50', '#FF9800']
for ax, label, color in zip(axes, labels, colors):
    vals = dfs[label]['Churn'].values
    ax.hist(vals, bins=60, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{label}\nμ={vals.mean():.4f}  σ={vals.std():.4f}', fontsize=11)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')

plt.suptitle('Prediction Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Helper: Rank Averaging ────────────────────────────────────────────────────
def rank_avg(pred_matrix, weights):
    """Convert each model's predictions to ranks, then weighted-average the ranks."""
    n_rows, n_models = pred_matrix.shape
    ranked = np.zeros_like(pred_matrix, dtype=float)
    for i in range(n_models):
        ranks = pred_matrix[:, i].argsort().argsort().astype(float)
        ranked[:, i] = ranks / (n_rows - 1)   # normalise to [0, 1]
    w = np.array(weights) / np.sum(weights)
    return ranked @ w

# ── Blend Strategies ──────────────────────────────────────────────────────────
CUSTOM_WEIGHTS = {
    'XGB-scaledpos': 2.0,   # current best (0.91445)
    'LGB-tuned-FE':  1.5,   # different algo — adds LGB diversity (0.91419)
    'XGB-50t-FE':    1.0,   # proven baseline — adds XGB diversity (0.91417)
}

w_custom = [CUSTOM_WEIGHTS[l] for l in labels]
w_equal  = [1.0] * len(labels)
w_score  = [s['public_score'] for s in SUBMISSIONS]

blend_configs = [
    ('equal_weighted',  w_equal,  False),
    ('equal_rank',      w_equal,  True),
    ('custom_weighted', w_custom, False),
    ('custom_rank',     w_custom, True),
    ('score_weighted',  w_score,  False),
    ('score_rank',      w_score,  True),
]

results = {}
print(f"{'Strategy':25s} | {'Range':22s} | Mean")
print('-' * 60)
for name, weights, use_rank in blend_configs:
    if use_rank:
        blended = rank_avg(preds, weights)
    else:
        w = np.array(weights) / np.sum(weights)
        blended = preds @ w
    results[name] = blended
    rng = f"[{blended.min():.4f}, {blended.max():.4f}]"
    print(f'{name:25s} | {rng:22s} | {blended.mean():.4f}')

print(f'\n✓ {len(results)} blend strategies computed.')

In [ ]:
# ── Inter-Blend Correlation ───────────────────────────────────────────────────
blend_names = list(results.keys())
blend_preds = np.column_stack([results[n] for n in blend_names])
blend_corr  = pd.DataFrame(
    np.corrcoef(blend_preds.T),
    index=blend_names, columns=blend_names
)

plt.figure(figsize=(10, 7))
sns.heatmap(blend_corr, annot=True, fmt='.5f', cmap='YlOrRd',
            vmin=0.99, vmax=1.0, linewidths=0.5)
plt.title('Correlation Between Blend Strategies', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save All Blend Submissions ────────────────────────────────────────────────
saved = []
for name, blended in results.items():
    out = base.copy()
    out['Churn'] = blended
    fname = f'submission_threeway_blend_{name}.csv'
    fpath = os.path.join(SUB_DIR, fname)
    out.to_csv(fpath, index=False)
    saved.append((fname, fpath))
    print(f'Saved → {fpath}')

print(f'\n✓ {len(saved)} blend files saved.')

In [ ]:
# ── Submit to Kaggle via local CLI ────────────────────────────────────────────
import subprocess

COMPETITION = 'playground-series-s6e3'

# Priority order: custom_weighted and custom_rank first, then equal_rank
SUBMIT_THESE = [
    ('submission_threeway_blend_custom_weighted.csv',
     'Threeway blend: XGB-scaledpos×2 + LGB-FE×1.5 + XGB-50t×1 | weighted avg'),
    ('submission_threeway_blend_custom_rank.csv',
     'Threeway blend: XGB-scaledpos×2 + LGB-FE×1.5 + XGB-50t×1 | rank avg'),
    ('submission_threeway_blend_equal_rank.csv',
     'Threeway blend: XGB-scaledpos + LGB-FE + XGB-50t | equal rank avg'),
]

for fname, message in SUBMIT_THESE:
    fpath = os.path.join(SUB_DIR, fname)
    if not os.path.exists(fpath):
        print(f'⚠️  File not found, skipping: {fpath}')
        continue
    print(f'\nSubmitting: {fname}')
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', COMPETITION,
         '-f', fpath,
         '-m', message],
        capture_output=True, text=True
    )
    print(result.stdout.strip())
    if result.stderr.strip():
        print('STDERR:', result.stderr.strip())

In [ ]:
# ── Check Submission Status ───────────────────────────────────────────────────
import subprocess, time

print('Waiting 15s for Kaggle to process...')
time.sleep(15)

result = subprocess.run(
    ['kaggle', 'competitions', 'submissions', '-c', 'playground-series-s6e3'],
    capture_output=True, text=True
)
print(result.stdout)